In [ ]:
#Bike sales dataset analysis
##1. Project Overview

This project performs an exploratory data analysis (EDA) on a bicycle product dataset to investigate pricing behavior, brand distribution, category segmentation, and potential data integrity issues.

The primary objectives of this analysis are:

Understand price distribution across brands and categories

Examine product trends across model years (2016-2019)

Assess data quality and identify structural inconsistencies

Evaluate the impact of missing reference mappings on analytical results

During the analysis, significant inconsistencies were identified in the brand reference table, particularly for model year 2019. A structured sensitivity analysis was conducted to measure how these issues affect overall business insights.

2. Data Loading

The analysis uses three datasets:

Products table: Contains product details including brand_id, category_id, model_year, and list_price.

Brands table: Reference table mapping brand_id to brand_name.

Categories table: Reference table mapping category_id to category_name.

The datasets were loaded and merged using left joins to preserve all product records while enriching them with brand and category names.

3. Data Cleaning

Data cleaning focused on:

Verifying data types

Checking for missing values

Validating foreign key consistency between products and reference tables

To ensure transparency during analysis:

Missing brand_name values were labeled as "Unmapped Brand"

Missing category_name values were labeled as "Unmapped Category"

This approach allowed structured analysis of missing data rather than dropping records prematurely.

4. Data Quality Assessment

A detailed data integrity audit revealed:

52% of products have unmapped brand references

42% of products have unmapped category references

100% of products in model year 2019 have unmapped brand references

Category mapping is fully recorded for 2019

These findings indicate a structural inconsistency in the brand reference table, particularly affecting the most recent model year.

The high percentage of unmapped brands suggests potential foreign key mismatches or incomplete dimension table coverage.

5. Brand Analysis

Brand-level pricing analysis revealed:

Unmapped Brand products have the highest average price (~2096)

Mapped brands (e.g., Electra, Surly, Haro) have significantly lower average prices

This indicates that missing brand references are disproportionately associated with premium-priced products.

Additionally:

Brand mapping integrity collapses entirely in 2019

Brand-based analysis for 2019 is therefore unreliable

This introduces substantial bias in overall pricing metrics when unmapped brands are included.

6️⃣ Category Analysis
6. Category Analysis

Category-level pricing analysis showed:

Road Bikes represent the premium segment (~3175 average price)

Mountain Bikes occupy a mid-range segment (~1649)

Children Bicycles represent the lowest price segment (~288)

Unmapped Category products fall within a mid-to-high price range (~1183)

Unlike brand mapping, category integrity remains structurally stable in 2019.

Category missingness appears distributed rather than structurally broken.

7. Sensitivity Analysis

To assess the impact of data inconsistencies, multiple analytical scenarios were compared:

Full dataset (including unmapped brands and 2019)

Dataset excluding Unmapped Brand

Dataset excluding model year 2019

Results show that:

Including unmapped brands significantly increases overall average price

Excluding 2019 stabilizes brand-level comparisons

Brand mapping inconsistencies materially affect pricing insights

This confirms that brand reference integrity is a critical analytical risk.

8. Conclusions & Limitations
Key Conclusions

Brand mapping inconsistencies represent the most critical data quality issue

Unmapped brands are strongly associated with high-priced products

Brand-based insights for 2019 are unreliable

Category segmentation remains structurally usable

Limitations

Brand reference table lacks full coverage for 2019

Foreign key integrity between products and brands is compromised

Results are sensitive to inclusion of unmapped records

Recommendations

Synchronize brand dimension table with product records

Implement foreign key validation checks

Introduce automated data quality monitoring for future datasets

In [ ]:
import pandas as pd
import numpy as np

In [28]:
brand=pd.read_csv('brands.csv')
categories=pd.read_csv('categories.csv')
products=pd.read_csv('products.csv')

In [29]:
brand.head(10)

,brand_id,brand_name
0,1,Electra
1,2,Haro
2,8,Surly


In [30]:
brand.shape

(3, 2)

In [31]:
brand.info()

<class 'pandas.DataFrame'>
RangeIndex: 3 entries, 0 to 2
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   brand_id    3 non-null      int64
 1   brand_name  3 non-null      str  
dtypes: int64(1), str(1)
memory usage: 180.0 bytes


In [32]:
categories.head(10)

,category_id,category_name
0,1,Children Bicycles
1,4,Cyclocross Bicycles
2,6,Mountain Bikes
3,7,Road Bikes


In [33]:
categories.shape

(4, 2)

In [34]:
categories.info()

<class 'pandas.DataFrame'>
RangeIndex: 4 entries, 0 to 3
Data columns (total 2 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   category_id    4 non-null      int64
 1   category_name  4 non-null      str  
dtypes: int64(1), str(1)
memory usage: 196.0 bytes


In [35]:
categories.duplicated().sum()

np.int64(0)

In [36]:
products.head(10)

,product_id,product_name,brand_id,category_id,model_year,list_price
0,1,Trek 820 - 2016,9,6,2016,379.99
1,2,Ritchey Timberwolf Frameset - 2016,5,6,2016,749.99
2,3,Surly Wednesday Frameset - 2016,8,6,2016,999.99
3,4,Trek Fuel EX 8 29 - 2016,9,6,2016,2899.99
4,5,Heller Shagamaw Frame - 2016,3,6,2016,1320.99
5,6,Surly Ice Cream Truck Frameset - 2016,8,6,2016,469.99
6,7,Trek Slash 8 27.5 - 2016,9,6,2016,3999.99
7,8,Trek Remedy 29 Carbon Frameset - 2016,9,6,2016,1799.99
8,9,Trek Conduit+ - 2016,9,5,2016,2999.99
9,12,Electra Townie Original 21D - 2016,1,3,2016,549.99


In [45]:
products.duplicated().sum()

np.int64(0)

In [55]:
products.isnull().sum() #the missing values in products table before merging with categories table and brand table

product_id      0
product_name    0
brand_id        0
category_id     0
model_year      0
list_price      0
dtype: int64

In [37]:
products.info()

<class 'pandas.DataFrame'>
RangeIndex: 311 entries, 0 to 310
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   product_id    311 non-null    int64  
 1   product_name  311 non-null    str    
 2   brand_id      311 non-null    int64  
 3   category_id   311 non-null    int64  
 4   model_year    311 non-null    int64  
 5   list_price    311 non-null    float64
dtypes: float64(1), int64(4), str(1)
memory usage: 14.7 KB


In [73]:
data=products.merge(brand, on='brand_id', how='left') #merging products table with brand table.
data=data.merge(categories, on='category_id', how='left') #merging products table with categories table to get the category name for each product.

In [74]:
data.head(15) #checking the first 15 rows of the merged data to see if the brand name and category name are added correctly. We can see that there are some missing values in the category_name column because there are some category_ids in the products table that are not present in the categories table. We will check for these missing category_ids in the next step.

,product_id,product_name,brand_id,category_id,model_year,list_price,brand_name,category_name
0,1,Trek 820 - 2016,9,6,2016,379.99,NaN,Mountain Bikes
1,2,Ritchey Timberwolf Frameset - 2016,5,6,2016,749.99,NaN,Mountain Bikes
2,3,Surly Wednesday Frameset - 2016,8,6,2016,999.99,Surly,Mountain Bikes
3,4,Trek Fuel EX 8 29 - 2016,9,6,2016,2899.99,NaN,Mountain Bikes
4,5,Heller Shagamaw Frame - 2016,3,6,2016,1320.99,NaN,Mountain Bikes
5,6,Surly Ice Cream Truck Frameset - 2016,8,6,2016,469.99,Surly,Mountain Bikes
6,7,Trek Slash 8 27.5 - 2016,9,6,2016,3999.99,NaN,Mountain Bikes
7,8,Trek Remedy 29 Carbon Frameset - 2016,9,6,2016,1799.99,NaN,Mountain Bikes
8,9,Trek Conduit+ - 2016,9,5,2016,2999.99,NaN,NaN
9,12,Electra Townie Original 21D - 2016,1,3,2016,549.99,Electra,NaN


In [77]:
data.duplicated().sum() #checking for duplicates in the merged data.

np.int64(0)

In [78]:
data.isnull().sum() #checking for missing vlaues in merged data.

product_id         0
product_name       0
brand_id           0
category_id        0
model_year         0
list_price         0
brand_name       162
category_name    132
dtype: int64

In [79]:
data.shape

(311, 8)

In [80]:
data[data['category_name'].isnull()]

,product_id,product_name,brand_id,category_id,model_year,list_price,brand_name,category_name
8,9,Trek Conduit+ - 2016,9,5,2016,2999.99,NaN,NaN
9,12,Electra Townie Original 21D - 2016,1,3,2016,549.99,Electra,NaN
10,13,Electra Cruiser 1 (24-Inch) - 2016,1,3,2016,269.99,Electra,NaN
11,14,Electra Girl's Hawaii 1 (16-inch) - 2015/2016,1,3,2016,269.99,Electra,NaN
12,15,Electra Moto 1 - 2016,1,3,2016,529.99,Electra,NaN
...,...,...,...,...,...,...,...,...
300,311,Electra Townie Commute 8D - 2018,1,2,2018,749.99,Electra,NaN
301,312,Electra Townie Commute 8D Ladies' - 2018,1,2,2018,699.99,Electra,NaN
302,313,Electra Townie Original 1 Ladies' - 2018,1,2,2018,449.99,Electra,NaN
303,314,Electra Townie Original 21D EQ Ladies' - 2018,1,2,2018,679.99,Electra,NaN


In [81]:
missing_category_ids = set(products['category_id']) - set(categories['category_id'])
missing_category_ids ##the category_ids that are in products but not in categories

{2, 3, 5}

In [82]:
products[products['category_id'].isin(missing_category_ids)]  #the product affected by the missing category_name.

,product_id,product_name,brand_id,category_id,model_year,list_price
8,9,Trek Conduit+ - 2016,9,5,2016,2999.99
9,12,Electra Townie Original 21D - 2016,1,3,2016,549.99
10,13,Electra Cruiser 1 (24-Inch) - 2016,1,3,2016,269.99
11,14,Electra Girl's Hawaii 1 (16-inch) - 2015/2016,1,3,2016,269.99
12,15,Electra Moto 1 - 2016,1,3,2016,529.99
...,...,...,...,...,...,...
300,311,Electra Townie Commute 8D - 2018,1,2,2018,749.99
301,312,Electra Townie Commute 8D Ladies' - 2018,1,2,2018,699.99
302,313,Electra Townie Original 1 Ladies' - 2018,1,2,2018,449.99
303,314,Electra Townie Original 21D EQ Ladies' - 2018,1,2,2018,679.99


In [96]:
#checking how impactful is the missing category_name
percentage_categories = 132 / len(data) * 100
percentage_categories


42.443729903536976

In [84]:
print(products['category_id'].nunique())
print(categories['category_id'].nunique())

6
4


In [85]:
data['category_name'] = data['category_name'].fillna("Unmapped Category")

In [86]:
data.head(10)

,product_id,product_name,brand_id,category_id,model_year,list_price,brand_name,category_name
0,1,Trek 820 - 2016,9,6,2016,379.99,NaN,Mountain Bikes
1,2,Ritchey Timberwolf Frameset - 2016,5,6,2016,749.99,NaN,Mountain Bikes
2,3,Surly Wednesday Frameset - 2016,8,6,2016,999.99,Surly,Mountain Bikes
3,4,Trek Fuel EX 8 29 - 2016,9,6,2016,2899.99,NaN,Mountain Bikes
4,5,Heller Shagamaw Frame - 2016,3,6,2016,1320.99,NaN,Mountain Bikes
5,6,Surly Ice Cream Truck Frameset - 2016,8,6,2016,469.99,Surly,Mountain Bikes
6,7,Trek Slash 8 27.5 - 2016,9,6,2016,3999.99,NaN,Mountain Bikes
7,8,Trek Remedy 29 Carbon Frameset - 2016,9,6,2016,1799.99,NaN,Mountain Bikes
8,9,Trek Conduit+ - 2016,9,5,2016,2999.99,NaN,Unmapped Category
9,12,Electra Townie Original 21D - 2016,1,3,2016,549.99,Electra,Unmapped Category


In [88]:
data[data['brand_name'].isnull()]

,product_id,product_name,brand_id,category_id,model_year,list_price,brand_name,category_name
0,1,Trek 820 - 2016,9,6,2016,379.99,NaN,Mountain Bikes
1,2,Ritchey Timberwolf Frameset - 2016,5,6,2016,749.99,NaN,Mountain Bikes
3,4,Trek Fuel EX 8 29 - 2016,9,6,2016,2899.99,NaN,Mountain Bikes
4,5,Heller Shagamaw Frame - 2016,3,6,2016,1320.99,NaN,Mountain Bikes
6,7,Trek Slash 8 27.5 - 2016,9,6,2016,3999.99,NaN,Mountain Bikes
...,...,...,...,...,...,...,...,...
306,317,Trek Checkpoint ALR 5 - 2019,9,7,2019,1999.99,NaN,Road Bikes
307,318,Trek Checkpoint ALR 5 Women's - 2019,9,7,2019,1999.99,NaN,Road Bikes
308,319,Trek Checkpoint SL 5 Women's - 2019,9,7,2019,2799.99,NaN,Road Bikes
309,320,Trek Checkpoint SL 6 - 2019,9,7,2019,3799.99,NaN,Road Bikes


In [90]:
missingbrand_ids = set(products['brand_id']) - set(brand['brand_id'])
missingbrand_ids ##the brand_ids that are in products but not in brand table.

{3, 4, 5, 6, 7, 9}

In [91]:
products[products['brand_id'].isin(missingbrand_ids)]

,product_id,product_name,brand_id,category_id,model_year,list_price
0,1,Trek 820 - 2016,9,6,2016,379.99
1,2,Ritchey Timberwolf Frameset - 2016,5,6,2016,749.99
3,4,Trek Fuel EX 8 29 - 2016,9,6,2016,2899.99
4,5,Heller Shagamaw Frame - 2016,3,6,2016,1320.99
6,7,Trek Slash 8 27.5 - 2016,9,6,2016,3999.99
...,...,...,...,...,...,...
306,317,Trek Checkpoint ALR 5 - 2019,9,7,2019,1999.99
307,318,Trek Checkpoint ALR 5 Women's - 2019,9,7,2019,1999.99
308,319,Trek Checkpoint SL 5 Women's - 2019,9,7,2019,2799.99
309,320,Trek Checkpoint SL 6 - 2019,9,7,2019,3799.99


In [92]:
data['brand_name'] = data['brand_name'].fillna("Unmapped Brand")

In [93]:
data

,product_id,product_name,brand_id,category_id,model_year,list_price,brand_name,category_name
0,1,Trek 820 - 2016,9,6,2016,379.99,Unmapped Brand,Mountain Bikes
1,2,Ritchey Timberwolf Frameset - 2016,5,6,2016,749.99,Unmapped Brand,Mountain Bikes
2,3,Surly Wednesday Frameset - 2016,8,6,2016,999.99,Surly,Mountain Bikes
3,4,Trek Fuel EX 8 29 - 2016,9,6,2016,2899.99,Unmapped Brand,Mountain Bikes
4,5,Heller Shagamaw Frame - 2016,3,6,2016,1320.99,Unmapped Brand,Mountain Bikes
...,...,...,...,...,...,...,...,...
306,317,Trek Checkpoint ALR 5 - 2019,9,7,2019,1999.99,Unmapped Brand,Road Bikes
307,318,Trek Checkpoint ALR 5 Women's - 2019,9,7,2019,1999.99,Unmapped Brand,Road Bikes
308,319,Trek Checkpoint SL 5 Women's - 2019,9,7,2019,2799.99,Unmapped Brand,Road Bikes
309,320,Trek Checkpoint SL 6 - 2019,9,7,2019,3799.99,Unmapped Brand,Road Bikes


In [94]:
data[data['brand_name'].isnull() & data['category_name'].isnull()]

,product_id,product_name,brand_id,category_id,model_year,list_price,brand_name,category_name


In [97]:
percentage_brand = 162 / len(data) * 100
percentage_brand


52.09003215434084

In [98]:
#Since 52% brand and 42% category are missing, i am going to make the new datasets that do not containing unmapped brand or unmapped category to make sure that the analysis is not affected by the missing values. I will also make a dataset that contains all the data including the unmapped brand and category to see how the missing values affect the analysis.

In [99]:
full_data = data.copy()

brand_valid = data[data['brand_name'] != "Unmapped Brand"]

category_valid = data[data['category_name'] != "Unmapped Category"]

fully_valid = data[
    (data['brand_name'] != "Unmapped Brand") &
    (data['category_name'] != "Unmapped Category")
]

In [103]:
full_data['brand_name'].value_counts()

brand_name
Unmapped Brand    162
Electra           118
Surly              21
Haro               10
Name: count, dtype: int64

In [104]:
brand_valid['brand_name'].value_counts()

brand_name
Electra    118
Surly       21
Haro        10
Name: count, dtype: int64

In [105]:
full_data['list_price'].mean()

np.float64(1487.7231832797427)

In [106]:
brand_valid['list_price'].mean()

np.float64(825.3991946308723)

In [107]:
full_data.groupby('brand_name')['list_price'].mean().sort_values(ascending=False)

brand_name
Unmapped Brand    2096.897716
Surly             1284.088095
Electra            761.006186
Haro               621.990000
Name: list_price, dtype: float64

In [108]:
full_data.groupby('model_year')['list_price'].mean()

model_year
2016     927.407917
2017    1226.435783
2018    1631.969747
2019    2583.323333
Name: list_price, dtype: float64

In [110]:
full_data.groupby('model_year')['brand_name'].apply(
    lambda x: (x == "Unmapped Brand").mean() * 100
)

model_year
2016     41.666667
2017     59.036145
2018     48.989899
2019    100.000000
Name: brand_name, dtype: float64

In [111]:
(full_data['category_name'] == "Unmapped Category") \
    .groupby(full_data['model_year']) \
    .mean() * 100

model_year
2016    54.166667
2017    37.349398
2018    44.444444
2019     0.000000
Name: category_name, dtype: float64

In [112]:
full_data.groupby('category_name')['list_price'].mean()

category_name
Children Bicycles     287.786610
Mountain Bikes       1649.757333
Road Bikes           3175.357333
Unmapped Category    1183.300152
Name: list_price, dtype: float64